# BTK Datathon 2026 - CatBoost Mixed Feature Experiment

Önemli notlar:
- Missing value doldurma train istatistikleriyle yapılır.
- CatBoost ana modeldir.
- CV çıktısında hem RMSE hem MSE yazdırılır.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from catboost import CatBoostRegressor

RANDOM_STATE = 42
TARGET = "career_success_score"
ID_COL = "student_id"

# İstediğin gibi aynı bırakıldı.
TRAIN_PATH = "train.csv"
TEST_PATH = "test_x.csv"

SUBMISSION_NAME = "results/catboost_mixed_features_submission.csv"

pd.set_option("display.max_columns", 200)

In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

assert TARGET in train_df.columns, f"{TARGET} train içinde yok."
assert ID_COL in train_df.columns, f"{ID_COL} train içinde yok."
assert ID_COL in test_df.columns, f"{ID_COL} test içinde yok."

print("Target range:", train_df[TARGET].min(), train_df[TARGET].max())
display(train_df.head())

Train shape: (10000, 47)
Test shape : (10000, 46)
Target range: 0.0 100.0


,student_id,application_year,age,graduation_year,department,university_tier,cgpa,english_exam_score,attendance_rate,failed_courses_count,target_role,coding_score,problem_solving_score,data_structures_score,sql_score,machine_learning_score,backend_score,frontend_score,cloud_score,devops_score,project_quality_score,real_client_project_count,internship_count,internship_duration_months,freelance_project_count,hackathon_count,hackathon_awards,portfolio_score,github_repo_count,github_avg_stars,open_source_contribution_count,linkedin_profile_score,cv_quality_score,technical_interview_score,hr_interview_score,communication_score,teamwork_score,leadership_score,presentation_score,certification_count,bootcamp_count,applications_sent,interviews_attended,hobby,preferred_social_media_platform,career_success_score,mentor_feedback_text
0,STU_000001,2021,21,2021,Computer Engineering,Tier 4,3.17,62.54,77.31,0,DevOps Engineer,73.28,71.11,52.91,84.980000,81.77,62.710000,71.570000,63.041897,69.952625,81.90,0,3,11.0,0,0,0,65.54,18,1.85,10.0,86.58,42.06,40.57,50.29,79.83,44.14,62.70,58.84,3,1,24,0,photography,LinkedIn,86.78,Proje kalitesi ve makine öğrenimi konusundaki ...
1,STU_000002,2024,20,2024,Computer Engineering,Tier 4,3.24,75.10,87.13,3,Backend Developer,63.12,78.90,61.81,37.450740,65.54,69.944694,60.830000,64.510000,57.940000,24.68,0,0,NaN,1,1,0,54.48,7,1.22,1.0,33.34,65.39,82.99,67.43,43.60,22.05,42.32,40.54,2,0,46,5,reading,YouTube,46.16,Kodlama ve problem çözme becerileri gelişmekte...
2,STU_000003,2024,28,2024,Electrical Electronics Engineering,Tier 4,3.00,68.53,95.64,1,Frontend Developer,100.00,86.44,83.62,85.440000,87.18,80.580000,96.433149,62.220000,81.750000,78.92,2,0,0.0,2,0,0,75.10,4,12.12,2.0,61.37,52.25,43.06,20.19,48.62,65.64,47.27,82.56,1,2,46,5,cinema,Reddit,84.08,İleri düzey frontend geliştirme becerileri ile...
3,STU_000004,2019,22,2018,Computer Engineering,Tier 1,2.82,54.85,77.80,2,Backend Developer,99.08,72.15,77.15,89.214871,69.49,85.751415,72.860000,73.680000,54.080000,54.93,0,1,9.0,0,1,0,82.40,4,2.96,3.0,45.15,24.12,32.06,28.00,59.84,3.89,78.69,85.05,2,4,49,7,running,Reddit,89.97,Güçlü bir kodlama yeteneği ve backend geliştir...
4,STU_000005,2026,22,2026,Computer Engineering,Tier 3,2.28,72.25,71.97,1,Product Analyst,92.65,91.15,84.51,70.700000,74.11,80.620000,86.830000,80.340000,87.560000,72.85,2,0,NaN,2,0,0,48.02,14,0.97,12.0,74.86,74.83,71.82,65.14,63.30,52.86,27.22,84.29,1,0,119,13,football,X,92.46,Ürün analizi alanına olan tutkusu ve makine öğ...


In [3]:
def add_missing_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    missing_cols = [
        "english_exam_score",
        "internship_duration_months",
        "portfolio_score",
        "github_avg_stars",
        "open_source_contribution_count",
        "linkedin_profile_score",
        "hr_interview_score"
    ]

    for col in missing_cols:
        if col in df.columns:
            df[f"{col}_was_missing"] = df[col].isna().astype(int)

    return df


def fill_missing_values(train: pd.DataFrame, test: pd.DataFrame):
    # Train'den öğrenilen istatistiklerle train/test eksik değerlerini doldurur.
    train = train.copy()
    test = test.copy()

    # 1) internship_duration_months: internship_count grubuna göre train median
    group_medians = train.groupby("internship_count")["internship_duration_months"].median()
    global_median = train["internship_duration_months"].median()

    for df in [train, test]:
        df["internship_duration_months"] = (
            df["internship_duration_months"]
            .fillna(df["internship_count"].map(group_medians))
            .fillna(global_median)
        )

    # 2) Mean ile doldurulacak kolonlar
    mean_fill_cols = [
        "english_exam_score",
        "linkedin_profile_score",
        "hr_interview_score",
        "portfolio_score"
    ]

    for col in mean_fill_cols:
        if col in train.columns:
            fill_value = train[col].mean()
            train[col] = train[col].fillna(fill_value)
            test[col] = test[col].fillna(fill_value)

    # 3) Median ile doldurulacak kolonlar
    median_fill_cols = [
        "github_avg_stars",
        "open_source_contribution_count"
    ]

    for col in median_fill_cols:
        if col in train.columns:
            fill_value = train[col].median()
            train[col] = train[col].fillna(fill_value)
            test[col] = test[col].fillna(fill_value)

    # 4) Kategorik / text kolonlarda güvenli doldurma
    object_cols = train.select_dtypes(include=["object", "category"]).columns.tolist()
    for col in object_cols:
        if col in test.columns:
            train[col] = train[col].fillna("Unknown")
            test[col] = test[col].fillna("Unknown")

    return train, test

In [4]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Tarih/yaş ilişkili feature'lar
    df["years_since_graduation"] = df["application_year"] - df["graduation_year"]
    df["age_at_graduation"] = df["age"] - df["years_since_graduation"]
    df["is_recent_graduate"] = (df["years_since_graduation"] <= 1).astype(int)

    # Teknik skor özetleri
    technical_skill_cols = [
        "coding_score", "problem_solving_score", "data_structures_score",
        "sql_score", "machine_learning_score", "backend_score",
        "frontend_score", "cloud_score", "devops_score"
    ]

    df["technical_skill_mean"] = df[technical_skill_cols].mean(axis=1)
    df["technical_skill_std"] = df[technical_skill_cols].std(axis=1)
    df["technical_skill_min"] = df[technical_skill_cols].min(axis=1)
    df["technical_skill_max"] = df[technical_skill_cols].max(axis=1)
    df["technical_skill_range"] = df["technical_skill_max"] - df["technical_skill_min"]

    # Data/AI odaklı skor
    df["data_ai_score"] = df[
        [
            "sql_score",
            "machine_learning_score",
            "problem_solving_score",
            "data_structures_score",
            "coding_score"
        ]
    ].mean(axis=1)

    # Software engineering odaklı skor
    df["software_engineering_score"] = df[
        [
            "backend_score",
            "frontend_score",
            "cloud_score",
            "devops_score",
            "coding_score",
            "data_structures_score"
        ]
    ].mean(axis=1)

    # Soft skill özeti
    soft_skill_cols = [
        "communication_score",
        "teamwork_score",
        "leadership_score",
        "presentation_score",
        "linkedin_profile_score",
        "cv_quality_score",
        "hr_interview_score"
    ]

    df["soft_skill_mean"] = df[soft_skill_cols].mean(axis=1)
    df["soft_skill_std"] = df[soft_skill_cols].std(axis=1)

    # Deneyim/aktivite özeti
    experience_cols = [
        "real_client_project_count",
        "internship_count",
        "freelance_project_count",
        "hackathon_count",
        "certification_count",
        "bootcamp_count"
    ]

    df["experience_total"] = df[experience_cols].sum(axis=1)

    # Senin mevcut güçlü project/portfolio feature'ın
    df["project_portfolio_score"] = (
        df["project_quality_score"]
        + df["portfolio_score"]
        + df["real_client_project_count"] * 5
        + df["freelance_project_count"] * 3
        + df["github_repo_count"] * 0.5
        + df["github_avg_stars"] * 0.5
        + df["open_source_contribution_count"] * 2
    )

    # denemeler.ipynb tarafındaki mantıklı project etkisi
    df["project_impact"] = (
        df["project_quality_score"] * (df["real_client_project_count"] + 1)
    )

    # Deneyim skoru
    df["experience_score"] = (
        df["internship_count"] * 10
        + df["internship_duration_months"] * 2
        + df["freelance_project_count"] * 5
        + df["real_client_project_count"] * 7
    )

    # Yarışma/hackathon feature'ları
    df["competition_score"] = (
        df["hackathon_count"] * 3
        + df["hackathon_awards"] * 10
    )

    df["hackathon_success_rate"] = (
        df["hackathon_awards"] / (df["hackathon_count"] + 1)
    )

    # Eğitim/öğrenme aktivitesi
    df["learning_activity_score"] = (
        df["certification_count"] * 2
        + df["bootcamp_count"] * 5
    )

    # Başvuru -> mülakat funnel'ı
    df["interview_conversion_rate"] = (
        df["interviews_attended"] / (df["applications_sent"] + 1)
    )

    df["applications_without_interview"] = (
        df["applications_sent"] - df["interviews_attended"]
    )

    df["applications_per_year_after_grad"] = (
        df["applications_sent"] / (df["years_since_graduation"] + 1)
    )

    df["interview_per_year_after_grad"] = (
        df["interviews_attended"] / (df["years_since_graduation"] + 1)
    )

    df["interview_score_mean"] = df[
        ["technical_interview_score", "hr_interview_score"]
    ].mean(axis=1)

    df["weighted_interview_score"] = (
        df["technical_interview_score"] * 0.6
        + df["hr_interview_score"] * 0.4
    )

    df["interview_success_score"] = (
        df["interview_conversion_rate"] * df["weighted_interview_score"]
    )

    # Github feature'ları
    df["github_impact_score"] = (
        df["github_repo_count"]
        + df["github_avg_stars"] * 2
        + df["open_source_contribution_count"] * 3
    )

    df["github_activity"] = (
        df["github_repo_count"] * df["github_avg_stars"]
    )

    df["github_per_repo_quality"] = (
        df["github_avg_stars"] / (df["github_repo_count"] + 1)
    )

    # Görünürlük / profil gücü feature'ları
    df["visibility_index"] = (
        df["linkedin_profile_score"] * 0.4
        + df["portfolio_score"] * 0.3
        + df["github_impact_score"] * 0.3
    )

    df["portfolio_visibility_score"] = (
        df["portfolio_score"] * 0.5
        + df["linkedin_profile_score"] * 0.3
        + df["cv_quality_score"] * 0.2
    )

    # Interaction feature'lar
    df["technical_x_project"] = (
        df["technical_skill_mean"] * df["project_portfolio_score"]
    )

    df["technical_x_experience"] = (
        df["technical_skill_mean"] * df["experience_score"]
    )

    df["soft_x_interview"] = (
        df["soft_skill_mean"] * df["weighted_interview_score"]
    )

    df["project_x_visibility"] = (
        df["project_portfolio_score"] * df["visibility_index"]
    )

    df["technical_x_visibility"] = (
        df["technical_skill_mean"] * df["visibility_index"]
    )

    df["internship_months_per_internship"] = (
        df["internship_duration_months"] / (df["internship_count"] + 1)
    )

    df["applications_pressure"] = (
        df["applications_sent"] / (df["interviews_attended"] + 1)
    )

    # Text feature'ları
    df["mentor_feedback_len"] = (
        df["mentor_feedback_text"]
        .fillna("")
        .astype(str)
        .str.len()
    )

    df["mentor_feedback_word_count"] = (
        df["mentor_feedback_text"]
        .fillna("")
        .astype(str)
        .str.split()
        .str.len()
    )

    df = df.replace([np.inf, -np.inf], np.nan)

    return df

In [5]:
def add_text_svd_features(
    train: pd.DataFrame,
    test: pd.DataFrame,
    text_col: str = "mentor_feedback_text",
    n_components: int = 10,
    max_features: int = 500
):
    # Mentor feedback text için TF-IDF + SVD numeric feature üretir. Fit sadece train üzerinde yapılır.
    train = train.copy()
    test = test.copy()

    if text_col not in train.columns or text_col not in test.columns:
        return train, test

    train_text = train[text_col].fillna("").astype(str)
    test_text = test[text_col].fillna("").astype(str)

    tfidf = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=2
    )

    train_tfidf = tfidf.fit_transform(train_text)
    test_tfidf = tfidf.transform(test_text)

    if train_tfidf.shape[1] <= 1:
        return train, test

    actual_components = min(n_components, train_tfidf.shape[1] - 1)

    svd = TruncatedSVD(
        n_components=actual_components,
        random_state=RANDOM_STATE
    )

    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    for i in range(actual_components):
        train[f"mentor_feedback_svd_{i}"] = train_svd[:, i]
        test[f"mentor_feedback_svd_{i}"] = test_svd[:, i]

    return train, test

In [6]:
USE_TEXT_SVD = True

train_tmp = add_missing_flags(train_df)
test_tmp = add_missing_flags(test_df)

train_tmp, test_tmp = fill_missing_values(train_tmp, test_tmp)

train_fe = add_features(train_tmp)
test_fe = add_features(test_tmp)

if USE_TEXT_SVD:
    train_fe, test_fe = add_text_svd_features(train_fe, test_fe)

print("Train missing after feature engineering:", train_fe.isna().sum().sum())
print("Test missing after feature engineering :", test_fe.isna().sum().sum())
print("Train shape after features:", train_fe.shape)
print("Test shape after features :", test_fe.shape)

display(train_fe.head())

Train missing after feature engineering: 0
Test missing after feature engineering : 0
Train shape after features: (10000, 104)
Test shape after features : (10000, 103)


,student_id,application_year,age,graduation_year,department,university_tier,cgpa,english_exam_score,attendance_rate,failed_courses_count,target_role,coding_score,problem_solving_score,data_structures_score,sql_score,machine_learning_score,backend_score,frontend_score,cloud_score,devops_score,project_quality_score,real_client_project_count,internship_count,internship_duration_months,freelance_project_count,hackathon_count,hackathon_awards,portfolio_score,github_repo_count,github_avg_stars,open_source_contribution_count,linkedin_profile_score,cv_quality_score,technical_interview_score,hr_interview_score,communication_score,teamwork_score,leadership_score,presentation_score,certification_count,bootcamp_count,applications_sent,interviews_attended,hobby,preferred_social_media_platform,career_success_score,mentor_feedback_text,english_exam_score_was_missing,internship_duration_months_was_missing,portfolio_score_was_missing,github_avg_stars_was_missing,open_source_contribution_count_was_missing,linkedin_profile_score_was_missing,hr_interview_score_was_missing,years_since_graduation,age_at_graduation,is_recent_graduate,technical_skill_mean,technical_skill_std,technical_skill_min,technical_skill_max,technical_skill_range,data_ai_score,software_engineering_score,soft_skill_mean,soft_skill_std,experience_total,project_portfolio_score,project_impact,experience_score,competition_score,hackathon_success_rate,learning_activity_score,interview_conversion_rate,applications_without_interview,applications_per_year_after_grad,interview_per_year_after_grad,interview_score_mean,weighted_interview_score,interview_success_score,github_impact_score,github_activity,github_per_repo_quality,visibility_index,portfolio_visibility_score,technical_x_project,technical_x_experience,soft_x_interview,project_x_visibility,technical_x_visibility,internship_months_per_internship,applications_pressure,mentor_feedback_len,mentor_feedback_word_count,mentor_feedback_svd_0,mentor_feedback_svd_1,mentor_feedback_svd_2,mentor_feedback_svd_3,mentor_feedback_svd_4,mentor_feedback_svd_5,mentor_feedback_svd_6,mentor_feedback_svd_7,mentor_feedback_svd_8,mentor_feedback_svd_9
0,STU_000001,2021,21,2021,Computer Engineering,Tier 4,3.17,62.54,77.31,0,DevOps Engineer,73.28,71.11,52.91,84.980000,81.77,62.710000,71.570000,63.041897,69.952625,81.90,0,3,11.0,0,0,0,65.54,18,1.85,10.0,86.58,42.06,40.57,50.29,79.83,44.14,62.70,58.84,3,1,24,0,photography,LinkedIn,86.78,Proje kalitesi ve makine öğrenimi konusundaki ...,0,0,0,0,0,0,0,0,21,1,70.147169,9.815953,52.91000,84.98,32.07000,72.810000,65.577420,60.634286,17.191141,7,177.365,81.90,52.0,0,0.0,11,0.000000,24,24.0,0.0,45.430,44.458,0.000000,51.70,33.30,0.097368,69.804,67.156,12441.652648,3647.652793,2695.679074,12380.78646,4896.552992,2.75,24.000000,225,30,0.389745,0.255982,0.232308,-0.177408,-0.014902,-0.145315,0.153264,0.007901,0.055527,0.102094
1,STU_000002,2024,20,2024,Computer Engineering,Tier 4,3.24,75.10,87.13,3,Backend Developer,63.12,78.90,61.81,37.450740,65.54,69.944694,60.830000,64.510000,57.940000,24.68,0,0,4.0,1,1,0,54.48,7,1.22,1.0,33.34,65.39,82.99,67.43,43.60,22.05,42.32,40.54,2,0,46,5,reading,YouTube,46.16,Kodlama ve problem çözme becerileri gelişmekte...,0,1,0,0,0,0,0,0,20,1,62.227270,11.118139,37.45074,78.90,41.44926,61.364148,63.025782,44.952857,16.383597,4,88.270,24.68,13.0,3,0.0,4,0.106383,41,46.0,5.0,75.210,76.766,8.166596,12.44,8.54,0.152500,33.412,50.320,5492.801166,808.954516,3450.851031,2949.27724,2079.137562,4.00,7.666667,259,33,0.382336,-0.149543,-0.041821,-0.198402,0.085593,0.065720,-0.080842,-0.017512,0.021671,-0.092733
2,STU_000003,2024,28,2024,Electrical Electronics Engineering,Tier 4,3.00,68.53,95.64,1,Frontend Developer,100.00,86.44,83.62,85.440000,87.18,80.580000,96.433149,62.220000,81.750000,78.92,2,0,0.0,2,0,0,75.10,4,12.12,2.0,61.37,52.25,43.06,20.19,48.62,65.64,47.27,82.56,1,2,46,5,cinema,Reddit,84.08,İleri düzey frontend geliştirme becerileri ile...,0,0,0,0,0,0,0,0,28,1,84.851461,10.685677,62.22000

In [7]:
y = train_fe[TARGET].copy()

DROP_COLS = [TARGET, ID_COL, "mentor_feedback_text"]

X = train_fe.drop(columns=[c for c in DROP_COLS if c in train_fe.columns])
X_test = test_fe.drop(columns=[c for c in [ID_COL, "mentor_feedback_text"] if c in test_fe.columns])

# Train ve test kolon hizalaması
X_test = X_test.reindex(columns=X.columns)

# CatBoost categorical feature'lar
cat_cols = [
    "department",
    "university_tier",
    "target_role",
    "hobby",
    "preferred_social_media_platform"
]
cat_cols = [c for c in cat_cols if c in X.columns]

for col in cat_cols:
    X[col] = X[col].fillna("Unknown").astype(str)
    X_test[col] = X_test[col].fillna("Unknown").astype(str)

cat_feature_indices = [X.columns.get_loc(c) for c in cat_cols]

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("Categorical columns:", cat_cols)
print("Categorical feature indices:", cat_feature_indices)

assert X.shape[1] == X_test.shape[1]

X shape: (10000, 101)
X_test shape: (10000, 101)
Categorical columns: ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
Categorical feature indices: [3, 4, 9, 42, 43]


In [8]:
def run_catboost_cv(cat_params, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), start=1):
        print(f"========== Fold {fold} ==========")

        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = CatBoostRegressor(**cat_params)
        model.fit(
            X_train,
            y_train,
            cat_features=cat_feature_indices,
            eval_set=(X_valid, y_valid),
            use_best_model=True
        )

        valid_pred = model.predict(X_valid)
        oof_preds[valid_idx] = valid_pred

        fold_rmse = mean_squared_error(y_valid, valid_pred) ** 0.5
        fold_scores.append(fold_rmse)

        print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

        test_preds += model.predict(X_test) / n_splits

    oof_rmse = mean_squared_error(y, oof_preds) ** 0.5
    oof_mse = mean_squared_error(y, oof_preds)

    print("========== CV RESULT ==========")
    print(f"Fold RMSE values: {[round(s, 5) for s in fold_scores]}")
    print(f"Mean RMSE: {np.mean(fold_scores):.5f}")
    print(f"Std RMSE : {np.std(fold_scores):.5f}")
    print(f"OOF RMSE : {oof_rmse:.5f}")
    print(f"OOF MSE  : {oof_mse:.5f}")

    return {
        "oof_preds": oof_preds,
        "test_preds": test_preds,
        "fold_scores": fold_scores,
        "oof_rmse": oof_rmse,
        "oof_mse": oof_mse
    }

In [9]:
cat_params = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "iterations": 1800,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 6,
    "random_seed": RANDOM_STATE,
    "od_type": "Iter",
    "od_wait": 150,
    "verbose": 200,
    "allow_writing_files": False
}

main_result = run_catboost_cv(cat_params, n_splits=5, seed=RANDOM_STATE)

oof_preds = main_result["oof_preds"]
test_preds = main_result["test_preds"]

========== Fold 1 ==========
0:	learn: 14.9845374	test: 14.9652614	best: 14.9652614 (0)	total: 177ms	remaining: 5m 18s
200:	learn: 8.8935629	test: 9.4252768	best: 9.4252768 (200)	total: 3.5s	remaining: 27.8s
400:	learn: 8.1720580	test: 9.1243808	best: 9.1243808 (400)	total: 6.69s	remaining: 23.3s
600:	learn: 7.6751795	test: 9.0415668	best: 9.0415668 (600)	total: 10.3s	remaining: 20.6s
800:	learn: 7.2533754	test: 9.0066873	best: 9.0060164 (793)	total: 13.5s	remaining: 16.9s
1000:	learn: 6.8907526	test: 8.9867752	best: 8.9864981 (998)	total: 16.7s	remaining: 13.4s
1200:	learn: 6.5459594	test: 8.9675530	best: 8.9664498 (1193)	total: 20.3s	remaining: 10.1s
1400:	learn: 6.2217498	test: 8.9666199	best: 8.9640743 (1369)	total: 23.3s	remaining: 6.65s
1600:	learn: 5.9358302	test: 8.9608737	best: 8.9595061 (1551)	total: 26.5s	remaining: 3.29s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 8.959370293
bestIteration = 1645

Shrink model to first 1646 iterations.
Fold 1 RMSE: 8.

In [10]:
# Tahminleri target aralığına sıkıştırıyoruz.
test_preds_clipped = np.clip(test_preds, 0, 100)

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_preds_clipped
})

submission.to_csv(SUBMISSION_NAME, index=False)

print(f"Saved: {SUBMISSION_NAME}")
print("Submission shape:", submission.shape)
print("Prediction min:", submission[TARGET].min())
print("Prediction max:", submission[TARGET].max())
print("Prediction mean:", submission[TARGET].mean())

display(submission.head())

Saved: results/catboost_mixed_features_submission.csv
Submission shape: (10000, 2)
Prediction min: 35.31010550800229
Prediction max: 100.0
Prediction mean: 76.08596443479698


,student_id,career_success_score
0,STU_010001,57.027560
1,STU_010002,72.505337
2,STU_010003,73.625881
3,STU_010004,94.714958
4,STU_010005,77.673864


## Opsiyonel: Parametre denemeleri

Aşağıdaki hücre varsayılan olarak kapalıdır. `RUN_PARAM_EXPERIMENTS = True` yaparsan 3 farklı CatBoost denemesi çalıştırır ve sonuçları karşılaştırır. Bu hücre daha uzun sürebilir.

In [11]:
RUN_PARAM_EXPERIMENTS = True

if RUN_PARAM_EXPERIMENTS:
    param_grid = {
        "mix_v1_depth6_lr003": {
            "loss_function": "RMSE",
            "eval_metric": "RMSE",
            "iterations": 1800,
            "learning_rate": 0.03,
            "depth": 6,
            "l2_leaf_reg": 6,
            "random_seed": RANDOM_STATE,
            "od_type": "Iter",
            "od_wait": 150,
            "verbose": 300,
            "allow_writing_files": False
        },
        "mix_v2_depth5_regularized": {
            "loss_function": "RMSE",
            "eval_metric": "RMSE",
            "iterations": 2500,
            "learning_rate": 0.025,
            "depth": 5,
            "l2_leaf_reg": 9,
            "random_seed": RANDOM_STATE,
            "od_type": "Iter",
            "od_wait": 200,
            "verbose": 300,
            "allow_writing_files": False
        },
        "mix_v3_depth7_slow": {
            "loss_function": "RMSE",
            "eval_metric": "RMSE",
            "iterations": 3000,
            "learning_rate": 0.02,
            "depth": 7,
            "l2_leaf_reg": 7,
            "random_seed": RANDOM_STATE,
            "od_type": "Iter",
            "od_wait": 250,
            "verbose": 300,
            "allow_writing_files": False
        }
    }

    experiment_results = {}

    for name, params in param_grid.items():
        print(f"#################### {name} ####################")
        result = run_catboost_cv(params, n_splits=5, seed=RANDOM_STATE)
        experiment_results[name] = result

    summary = pd.DataFrame([
        {
            "experiment": name,
            "oof_rmse": result["oof_rmse"],
            "oof_mse": result["oof_mse"],
            "mean_fold_rmse": np.mean(result["fold_scores"]),
            "std_fold_rmse": np.std(result["fold_scores"]),
        }
        for name, result in experiment_results.items()
    ]).sort_values("oof_mse")

    display(summary)

#################### mix_v1_depth6_lr003 ####################
========== Fold 1 ==========
0:	learn: 14.9845374	test: 14.9652614	best: 14.9652614 (0)	total: 18ms	remaining: 32.4s
300:	learn: 8.5120528	test: 9.2425928	best: 9.2425928 (300)	total: 4.81s	remaining: 24s
600:	learn: 7.6751795	test: 9.0415668	best: 9.0415668 (600)	total: 9.67s	remaining: 19.3s
900:	learn: 7.0645512	test: 8.9998157	best: 8.9982869 (865)	total: 15.3s	remaining: 15.2s
1200:	learn: 6.5459594	test: 8.9675530	best: 8.9664498 (1193)	total: 20.4s	remaining: 10.2s
1500:	learn: 6.0731006	test: 8.9636420	best: 8.9629441 (1487)	total: 25.3s	remaining: 5.03s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 8.959370293
bestIteration = 1645

Shrink model to first 1646 iterations.
Fold 1 RMSE: 8.95937
========== Fold 2 ==========
0:	learn: 14.9297072	test: 15.1848746	best: 15.1848746 (0)	total: 13.4ms	remaining: 24s
300:	learn: 8.4588598	test: 9.4017474	best: 9.4017474 (300)	total: 4.59s	remaining: 22.9s
6

,experiment,oof_rmse,oof_mse,mean_fold_rmse,std_fold_rmse
0,mix_v1_depth6_lr003,8.920282,79.571425,8.917578,0.219597
2,mix_v3_depth7_slow,8.921296,79.589529,8.918337,0.229760
1,mix_v2_depth5_regularized,8.922162,79.604978,8.919016,0.236914


In [13]:
best_name = summary.iloc[0]["experiment"]
best_result = experiment_results[best_name]

best_test_preds = np.clip(best_result["test_preds"], 0, 100)

best_submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: best_test_preds
})

best_submission.to_csv("results/best_param_submission.csv", index=False)

print(f"Best experiment: {best_name}")
print(f"Saved: results/best_param_submission.csv")
display(best_submission.head())

Best experiment: mix_v1_depth6_lr003
Saved: results/best_param_submission.csv


,student_id,career_success_score
0,STU_010001,57.027560
1,STU_010002,72.505337
2,STU_010003,73.625881
3,STU_010004,94.714958
4,STU_010005,77.673864


## Opsiyonel: Deneme sonuçlarından ensemble submission

Parametre denemelerini çalıştırdıysan, aşağıdaki hücre en iyi modellerin ortalamasını alır. Çalıştırmadan önce `experiment_results` objesinin oluşmuş olması gerekir.

In [12]:
RUN_EXPERIMENT_ENSEMBLE = True

if RUN_EXPERIMENT_ENSEMBLE:
    assert "experiment_results" in globals(), "Önce RUN_PARAM_EXPERIMENTS=True ile parametre denemelerini çalıştır."

    # Basit eşit ağırlıklı ensemble
    ensemble_oof = np.mean(
        [result["oof_preds"] for result in experiment_results.values()],
        axis=0
    )

    ensemble_test = np.mean(
        [result["test_preds"] for result in experiment_results.values()],
        axis=0
    )

    ensemble_rmse = mean_squared_error(y, ensemble_oof) ** 0.5
    ensemble_mse = mean_squared_error(y, ensemble_oof)

    print(f"Ensemble OOF RMSE: {ensemble_rmse:.5f}")
    print(f"Ensemble OOF MSE : {ensemble_mse:.5f}")

    ensemble_test_clipped = np.clip(ensemble_test, 0, 100)

    ensemble_submission = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        TARGET: ensemble_test_clipped
    })

    ensemble_submission_name = "results/catboost_mixed_features_ensemble_submission.csv"
    ensemble_submission.to_csv(ensemble_submission_name, index=False)

    print(f"Saved: {ensemble_submission_name}")
    display(ensemble_submission.head())

Ensemble OOF RMSE: 8.90299
Ensemble OOF MSE : 79.26328
Saved: results/catboost_mixed_features_ensemble_submission.csv


,student_id,career_success_score
0,STU_010001,56.987898
1,STU_010002,72.529303
2,STU_010003,73.360383
3,STU_010004,94.995186
4,STU_010005,77.792660
